# PKG geo validation (F2 follow-up): completeness, evidence by name class, and an outside test of `pkg_cpty_geo`

**PNC Treasury Management · DSI · Payment Knowledge Graph** · read-only on the published geo tables; writes work files
(deleted at the end) and CSVs in `OUT_DIR`, no tables.

| § | question |
|---|---|
| §1 | is every published edge month in `pkg_cust_geo_monthly`, built with the current signature? |
| §2 | which kinds of outside entities get located: `name_class` × `evidence_class`, and evidence class by number of PNC customers |
| §3 | outside entities that are the same business as a PNC organisation customer (cleaned name, unique on both sides) |
| §4 | `locate()` on those entities **with the matched customer left out of every neighbour's cloud**, compared with the customer's address: error, hit rates and radius coverage by evidence class; same for the published estimate; by truth tier (v1.1) |
| §5 | (v1.1) customer self-scoring again on a sample, with a **strict leave-one-out**: every point within `STRICT_KM` of the target removed from each neighbour's cloud (affiliates, households, other records of the same business), not only the target |

Why §4 matters: every accuracy figure so far comes from PNC customers scored against their own addresses. Outside
entities are what `pkg_cpty_geo` is for, and this is the only ground truth for them. Caveats: a matched entity can be a
branch or a second account of a multi-site business whose registered address is the head office, so errors here are
partly truth error (split by `pin_representative`, as always); and matching by name favours businesses with
distinctive names.

## Settings

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------
DB = "bdahd01p_dlcdi1_cdi_tm"
REF = "_v5"
T_CUST_GEO = f"{DB}.pkg_cust_geo"
T_CPTY_GEO = f"{DB}.pkg_cpty_geo"
T_MONTHLY  = f"{DB}.pkg_cust_geo_monthly"
T_CUST     = f"{DB}.pkg_cust_dim{REF}"
T_EDGE_M   = f"{DB}.pkg_edge_monthly"
T_MAP      = f"{DB}.pkg_cpty_entity_map{REF}"
T_ENT      = f"{DB}.pkg_cpty_entity_dim{REF}"
GEO_DIR    = "../metrics/geo"                 # QA files of pkg_geo_metrics (radius_calibration.csv is read)
OUT_DIR    = "../metrics/geo/validation"

GEO_SIG_EXPECTED = None        # None = the signature most months carry; or paste the §1 signature of pkg_geo_metrics
# ---- must equal pkg_geo_metrics (v1.0.1)
KERNEL_MONTHS = 12
MIN_CP_BW, MIN_CP_BW_TRUST, COLOC_WIDTH_KM, K_MAX = 2, 5, 0.1, 20
REP_GAP_KM = 25
CORE_SHARE = 0.80
TRUTH_STATUSES = ["ok", "shared_building", "low_precision"]
# ---- name match
MATCH_NAME_CLASSES = ["business"]          # entity name classes eligible for the match
LEGAL_WORDS = ["INC", "INCORPORATED", "LLC", "L L C", "LC", "CORP", "CORPORATION", "CO", "COMPANY", "LTD", "LIMITED",
               "LP", "LLP", "PLLC", "PC", "PA", "THE", "NA", "N A", "DBA", "GROUP", "HOLDINGS"]
KEY_MIN_CHARS = 6                          # cleaned name at least this long (letters and digits)
# ---- v1.1 diagnostics
FAR_KM = 500                               # an error beyond this is the signature of a wrong truth address
STRICT_KM = 1.0                            # §5: cloud points this close to the target are removed with it
STRICT_PCT = 2                             # §5: % of located customers re-scored (hash)
STRICT_CELL_DEG = 0.02                     # §5: grid used to find the nearby cloud points (must exceed STRICT_KM)
SEED = 17

# ---- cluster (no effect on the numbers)
SHUFFLE_PARTITIONS = 2000
RESOURCE_PROFILE = "acc"
RESOURCES = {
    "acc":      {"spark.executor.memory": "4g", "spark.executor.memoryOverhead": "1g", "spark.executor.cores": "1",
                 "spark.driver.memory": "4g", "spark.yarn.am.memory": "4g", "spark.yarn.am.memoryOverhead": "1g",
                 "spark.dynamicAllocation.maxExecutors": "512", "spark.driver.maxResultSize": "2g"},
    "standard": {"spark.executor.memory": "8g", "spark.executor.memoryOverhead": "3g", "spark.executor.cores": "2",
                 "spark.driver.memory": "4g", "spark.yarn.am.memory": "4g", "spark.yarn.am.memoryOverhead": "1g",
                 "spark.dynamicAllocation.maxExecutors": "512", "spark.driver.maxResultSize": "4g"},
}
HDFS_REPLICATION = 2
VAL_VERSION = "geo-val-v1.1"

## Setup

In [ ]:
# ---- Setup ---------------------------------------------------------------------------------------
import numpy as np
import pandas as pd
import warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
warnings.filterwarnings("ignore")

import os, math, time, json, datetime as dt, html as _html
from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F

_b = SparkSession.builder.appName("pkg_geo_validation").config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
for _k, _v in RESOURCES[RESOURCE_PROFILE].items():
    _b = _b.config(_k, _v)
_b = (_b.config("spark.hadoop.dfs.replication", str(HDFS_REPLICATION))
        .config("spark.dynamicAllocation.enabled", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
        .config("spark.ui.showConsoleProgress", "false")
        .enableHiveSupport())
spark = _b.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.sparkContext.setLogLevel("ERROR")
for _n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
           "org.apache.spark.network", "io.netty"):
    try:
        _L = spark._jvm.org.apache.logging.log4j
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
    except Exception:
        pass
HCONF = spark._jsc.hadoopConfiguration()
HCONF.set("dfs.replication", str(HDFS_REPLICATION))
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 80)

def tbl(pdf, pct=(), dec=()):
    pdf = pdf.copy()
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                   fmt[c] = "{:.1%}"
        elif c in dec:                                 fmt[c] = "{:,.2f}"
        elif pd.api.types.is_bool_dtype(pdf[c]):       continue
        elif pd.api.types.is_integer_dtype(pdf[c]):    fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):      fmt[c] = "{:,.1f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:
        sty = sty.hide(axis="index")
    except AttributeError:
        sty = sty.hide_index()
    sty = sty.set_table_styles([{"selector": "th, td", "props": "padding: 2px 8px; font-size: 12px;"}])
    return sty.to_html()
def sec(title, body=""):
    return f"<h4 style='margin:14px 0 4px 0'>{_html.escape(title)}</h4>{body}"
def para(text, color=None):
    st = f"; color:{color}" if color else ""
    return f"<p style='margin:4px 0; font-size:13px{st}'>{_html.escape(text)}</p>"
def out(*parts):
    display(HTML("<div>" + "".join(p for p in parts if p) + "</div>"))

JPath = spark._jvm.org.apache.hadoop.fs.Path
def fs_for(p):
    return JPath(p).getFileSystem(HCONF)
def exists(p):
    return fs_for(p).exists(JPath(p))
def delete(p):
    return fs_for(p).delete(JPath(p), True) if exists(p) else False
def read_json(p):
    if not exists(p):
        return None
    s = fs_for(p).open(JPath(p))
    try:
        return json.loads(spark._jvm.org.apache.commons.io.IOUtils.toString(s, "UTF-8"))
    finally:
        s.close()
def table_location(t):
    return [r[1] for r in spark.sql(f"DESCRIBE TABLE EXTENDED {t}").collect() if r[0] == "Location"][0].rstrip("/")
def partitions(table, key):
    vals = set()
    try:
        for r in spark.sql(f"SHOW PARTITIONS {table}").collect():
            for kv in r[0].split("/"):
                if kv.startswith(key + "="):
                    vals.add(kv.split("=", 1)[1])
    except Exception:
        vals = {str(r[0]) for r in spark.table(table).select(key).distinct().collect()}
    return sorted(v for v in vals if v and v != "None")
def month_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"
def need(table, cols):
    dt_ = dict(spark.table(table).dtypes)
    miss = [c for c in cols if c not in dt_]
    assert not miss, f"{table}: missing {miss}; has {sorted(dt_)}"
    return dt_

R_EARTH_KM = 6371.0088
def unit(lat, lon):
    la, lo = F.radians(lat), F.radians(lon)
    return F.cos(la) * F.cos(lo), F.cos(la) * F.sin(lo), F.sin(la)
def chord_km(x1, y1, z1, x2, y2, z2):
    c = F.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2 + (z1 - z2) ** 2)
    return 2 * R_EARTH_KM * F.asin(F.least(F.lit(1.0), c / 2))
def latlon(x, y, z):
    return F.degrees(F.atan2(z, F.sqrt(x * x + y * y))), F.degrees(F.atan2(y, x))
def width_km(rbar):
    return R_EARTH_KM * F.sqrt(F.greatest(F.lit(0.0), 2 * (1 - rbar)))

DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
WORK = f"{DB_LOC}/_tmp_pkg_geo_validation"
os.makedirs(OUT_DIR, exist_ok=True)
T_START = time.time()
out(sec(f"Spark {spark.version} · profile {RESOURCE_PROFILE} · {VAL_VERSION}"))

## §1 Completeness of the monthly block

Every published edge month (with its `_PKG_BUILD` marker) against the folders of `pkg_cust_geo_monthly`: present, the
marker's signature and snapshot, rows. A month missing or built with another signature is rebuilt by running
`pkg_geo_metrics` again (Run All; finished months are skipped).

In [ ]:
# §1  Completeness
E_LOC = table_location(T_EDGE_M)
EDGE_MONTHS = [m for m in partitions(T_EDGE_M, "month") if exists(f"{E_LOC}/month={m}/_PKG_BUILD")]
M_LOC = table_location(T_MONTHLY)
rows = []
for m in EDGE_MONTHS:
    mk = read_json(f"{M_LOC}/month={m}/_GEO_BUILD")
    rows.append({"month": m, "folder": exists(f"{M_LOC}/month={m}/_SUCCESS"),
                 "signature": (mk or {}).get("sig"), "snapshot": (mk or {}).get("snapshot"),
                 "built_at": (mk or {}).get("built_at"), "rows": (mk or {}).get("rows")})
C1 = pd.DataFrame(rows)
sig_main = GEO_SIG_EXPECTED or (C1.signature.dropna().mode().iloc[0] if C1.signature.notna().any() else None)
C1["status"] = np.where(~C1.folder | C1.signature.isna(), "MISSING",
                        np.where(C1.signature != sig_main, "other signature", "ok"))
reg = set(partitions(T_MONTHLY, "month"))
C1["registered"] = C1.month.isin(reg)
C1.to_csv(f"{OUT_DIR}/completeness.csv", index=False)
bad = C1[C1.status != "ok"]
out(sec(f"§1 Monthly block · {len(EDGE_MONTHS)} edge months · {int((C1.status == 'ok').sum())} ok · "
        f"{len(bad)} missing or other signature · main signature {sig_main}"),
    para("Every month is in place." if bad.empty else
         f"Not in place: {', '.join(bad.month)}. Run pkg_geo_metrics again with MONTHS_BACK = None (finished months are skipped).",
         None if bad.empty else "#a00"),
    tbl(C1[["month", "status", "registered", "rows", "snapshot", "built_at", "signature"]]))

## §2 Which outside entities get located

`pkg_cpty_geo` by `name_class` and by number of PNC neighbours (customers it traded with in the kernel window), with the
entity's lifetime dollars from the entity table (`amt_lifetime`; dollars over all months, not the window).

In [ ]:
# §2  Evidence by name class and by number of neighbours
need(T_CPTY_GEO, ["ent_id", "evidence_class", "n_neighbours", "name_class", "r50_km"])
ent_cols = need(T_ENT, ["ent_id", "ent_name", "name_class"])
cg = spark.table(T_CPTY_GEO)
amt = (spark.table(T_ENT).select(F.col("ent_id").cast("string").alias("ent_id"), F.col("amt_lifetime").cast("double").alias("usd"))
       if "amt_lifetime" in ent_cols else spark.table(T_ENT).select(F.col("ent_id").cast("string").alias("ent_id"), F.lit(None).cast("double").alias("usd")))
cg2 = cg.withColumn("ent_id", F.col("ent_id").cast("string")).join(amt, "ent_id", "left")
CLS = ["PIN", "METRO", "REGION", "NONE", "NO_EVIDENCE"]
def pivot_share(df, by):
    p = (df.groupBy(by, "evidence_class").agg(F.count("*").alias("n"), F.sum("usd").alias("usd")).toPandas())
    n = p.pivot_table(index=by, columns="evidence_class", values="n", aggfunc="sum", fill_value=0)
    n = n.reindex(columns=[c for c in CLS if c in n.columns])
    tot = n.sum(axis=1)
    s = n.div(tot, axis=0)
    u = p.groupby(by)["usd"].sum()
    uloc = p[p.evidence_class.isin(["PIN", "METRO"])].groupby(by)["usd"].sum()
    r = pd.DataFrame({by: n.index, "entities": tot.values.astype(int)})
    for c in n.columns:
        r[f"{c}"] = s[c].values
    r["located_PIN_METRO"] = (n.get("PIN", 0) + n.get("METRO", 0)).values / tot.values
    r["usd_share_located"] = (uloc.reindex(n.index).fillna(0) / u.reindex(n.index).replace(0, np.nan)).values
    return r.sort_values("entities", ascending=False)
NC = pivot_share(cg2.withColumn("name_class", F.coalesce("name_class", F.lit("(none)"))), "name_class")
nb = F.col("n_neighbours")
cg3 = cg2.withColumn("pnc_neighbours", F.when(nb.isNull(), "0").when(nb == 1, "1").when(nb < 5, "2–4")
                     .when(nb < 20, "5–19").when(nb < 100, "20–99").otherwise("100+"))
NB = pivot_share(cg3, "pnc_neighbours")
NB["_o"] = NB.pnc_neighbours.map({k: i for i, k in enumerate(["0", "1", "2–4", "5–19", "20–99", "100+"])})
NB = NB.sort_values("_o").drop(columns="_o")
NC.to_csv(f"{OUT_DIR}/evidence_by_name_class.csv", index=False); NB.to_csv(f"{OUT_DIR}/evidence_by_neighbours.csv", index=False)
pc = [c for c in NC.columns if c not in ("name_class", "entities")]
out(sec("§2 Evidence class by name class (share of entities; dollars = lifetime)"), tbl(NC, pct=pc),
    sec("Evidence class by number of PNC neighbours in the window"), tbl(NB, pct=[c for c in NB.columns if c not in ("pnc_neighbours", "entities")]))

## §3 Outside entities that are the same business as a PNC organisation customer

Cleaned name: upper case, `&` → AND, punctuation removed, legal words dropped (`LEGAL_WORDS`), spaces collapsed; at least
`KEY_MIN_CHARS` letters or digits. A match needs the key to be unique among eligible entities (`MATCH_NAME_CLASSES`) and
unique among PNC organisation customers, and the customer to have a usable truth address (`TRUTH_STATUSES`). PNC's own
accounts and named large accounts (`name_group`) are left out. The entity table's `pnc_match_n` is shown for comparison.

In [ ]:
# §3  Name match
t0 = time.time()
cust_cols = need(T_CUST, ["mdm_id", "customer_name", "party_type"])
legal = "|".join(sorted((w.replace(" ", r"\s") for w in LEGAL_WORDS), key=len, reverse=True))
def clean_key(c):
    s = F.upper(F.coalesce(c, F.lit("")))
    s = F.regexp_replace(s, "&", " AND ")
    s = F.regexp_replace(s, r"[^A-Z0-9 ]", " ")
    s = F.regexp_replace(s, rf"(^| )({legal})(?= |$)", " ")
    s = F.trim(F.regexp_replace(s, r"\s+", " "))
    return s
def plain(c):
    """Upper case, punctuation removed, spaces collapsed; legal words kept (for the exact-name tier)."""
    s = F.regexp_replace(F.upper(F.coalesce(c, F.lit(""))), "&", " AND ")
    return F.trim(F.regexp_replace(F.regexp_replace(s, r"[^A-Z0-9 ]", " "), r"\s+", " "))
ents = (spark.table(T_ENT).where(F.col("name_class").isin(MATCH_NAME_CLASSES))
          .select(F.col("ent_id").cast("string").alias("ent_id"), "ent_name",
                  *(["pnc_match_n"] if "pnc_match_n" in ent_cols else []))
          .withColumn("key", clean_key(F.col("ent_name")))
          .where(F.length(F.regexp_replace("key", " ", "")) >= KEY_MIN_CHARS))
ents_u = ents.join(ents.groupBy("key").count().where("count = 1").select("key"), "key", "left_semi")
cflt = F.col("party_type") == "O"
if "is_pnc_own" in cust_cols:
    cflt = cflt & ~F.coalesce(F.col("is_pnc_own").cast("boolean"), F.lit(False))
if "name_group" in cust_cols:
    cflt = cflt & F.col("name_group").isNull()
custs = (spark.table(T_CUST).where(cflt)
           .select(F.trim(F.col("mdm_id")).alias("mdm_id"), "customer_name")
           .withColumn("key", clean_key(F.col("customer_name")))
           .where(F.length(F.regexp_replace("key", " ", "")) >= KEY_MIN_CHARS))
custs_u = custs.join(custs.groupBy("key").count().where("count = 1").select("key"), "key", "left_semi")
geo = (spark.table(T_CUST_GEO).where(F.col("geo_status").isin(TRUTH_STATUSES))
         .select(F.trim(F.col("mdm_id")).alias("mdm_id"), "lat", "lon", "state", "geo_status",
                 "geo_registered_vs_flow_km", "pin_representative"))
x, y, z = unit(F.col("lat"), F.col("lon"))
MATCH = (ents_u.join(custs_u, "key").join(geo, "mdm_id")
           .withColumn("name_exact", plain(F.col("ent_name")) == plain(F.col("customer_name")))
           .withColumn("cx", x).withColumn("cy", y).withColumn("cz", z))
MATCH.write.mode("overwrite").parquet(f"{WORK}/match"); MATCH = spark.read.parquet(f"{WORK}/match")
mm = MATCH.agg(F.count("*").alias("n"), *( [F.sum((F.coalesce("pnc_match_n", F.lit(0)) > 0).cast("long")).alias("flag")]
                                           if "pnc_match_n" in ent_cols else [F.lit(None).alias("flag")])).first()
counts = pd.DataFrame([
    {"step": "eligible entities with a usable cleaned name", "value": ents.count()},
    {"step": "… with a unique key", "value": ents_u.count()},
    {"step": "organisation customers with a unique key (no PNC own, no named large accounts)", "value": custs_u.count()},
    {"step": "matched pairs with a usable truth address", "value": mm["n"]},
    {"step": "… of which the entity table also flags (pnc_match_n > 0)", "value": mm["flag"] if mm["flag"] is not None else -1},
])
SAMPLE = MATCH.orderBy(F.rand(SEED)).limit(25).select("ent_name", "customer_name", "state", "geo_status").toPandas()
out(sec(f"§3 Name match · {(time.time() - t0) / 60:.1f} min"), tbl(counts),
    sec("25 random matched pairs (check they are the same business)"), tbl(SAMPLE))

## §4 Outside test: `locate()` with the matched customer left out

Neighbours of each matched entity = customers that traded with it in the kernel window (the last `KERNEL_MONTHS`
months). Each neighbour's kernel is read from `pkg_cust_geo` (`k_sx`, `k_sy`, `k_sz`, `k_n_cp`); when the neighbour also
traded with the matched customer in the window, that customer's point is subtracted from the kernel (exact
leave-one-out), and the matched customer itself is never a neighbour. Then the same guards, tightest `K_MAX`,
inverse-variance combination and radius calibration (`radius_calibration.csv`) as `pkg_geo_metrics`. Compared with the
customer's address; the published estimate in `pkg_cpty_geo` is scored too (its gap to the leave-one-out result is the
leak a same-business customer gives).

In [ ]:
# §4  Outside test
t0 = time.time()
LAST = EDGE_MONTHS[-1]
K_MONTHS = [m for m in EDGE_MONTHS if m > month_add(LAST, -KERNEL_MONTHS)]
MATCH = spark.read.parquet(f"{WORK}/match")
mp = (spark.table(T_MAP).select(F.trim(F.col("cpty_id")).alias("cpty"), F.col("ent_l2").cast("string").alias("ent_id"))
        .groupBy("cpty").agg(F.min("ent_id").alias("ent_id"))
        .join(MATCH.select("ent_id"), "ent_id", "left_semi"))
E = (spark.table(T_EDGE_M).where(F.col("month").isin(K_MONTHS) & ~F.coalesce(F.col("is_self").cast("boolean"), F.lit(False)))
       .select(F.trim(F.col("src_id")).alias("src"), "src_kind", F.trim(F.col("dst_id")).alias("dst"), "dst_kind"))
side = (E.where((F.col("src_kind") == "CUST") & (F.col("dst_kind") == "CPTY")).select(F.col("src").alias("j"), F.col("dst").alias("cpty"))
        .unionByName(E.where((F.col("src_kind") == "CPTY") & (F.col("dst_kind") == "CUST")).select(F.col("dst").alias("j"), F.col("src").alias("cpty"))))
NB_ = side.join(F.broadcast(mp) if mp.count() < 2_000_000 else mp, "cpty").select("ent_id", "j").distinct()
NB_ = NB_.join(MATCH.select("ent_id", F.col("mdm_id").alias("c")), "ent_id")
DIRECT = NB_.where(F.col("j") == F.col("c")).select("ent_id").distinct().withColumn("direct_link", F.lit(True))
DIRECT.write.mode("overwrite").parquet(f"{WORK}/direct"); DIRECT = spark.read.parquet(f"{WORK}/direct")
NB_ = NB_.where(F.col("j") != F.col("c"))
NB_.write.mode("overwrite").parquet(f"{WORK}/nb"); NB_ = spark.read.parquet(f"{WORK}/nb")
cset = MATCH.select(F.col("mdm_id").alias("c")).distinct()
cc = (E.where((F.col("src_kind") == "CUST") & (F.col("dst_kind") == "CUST"))
        .select(F.col("src").alias("j"), F.col("dst").alias("c"))
        .unionByName(E.where((F.col("src_kind") == "CUST") & (F.col("dst_kind") == "CUST")).select(F.col("dst").alias("j"), F.col("src").alias("c")))
        .join(cset, "c", "left_semi").distinct().withColumn("j_has_c", F.lit(True)))
K = spark.table(T_CUST_GEO).select(F.trim(F.col("mdm_id")).alias("j"), "k_sx", "k_sy", "k_sz", "k_n_cp").where(F.col("k_n_cp").isNotNull())
P = (NB_.join(K, "j").join(cc, ["j", "c"], "left")
       .join(MATCH.select("ent_id", "cx", "cy", "cz"), "ent_id")
       .withColumn("loo", F.coalesce("j_has_c", F.lit(False)))
       .withColumn("kx", F.when(F.col("loo"), F.col("k_sx") - F.col("cx")).otherwise(F.col("k_sx")))
       .withColumn("ky", F.when(F.col("loo"), F.col("k_sy") - F.col("cy")).otherwise(F.col("k_sy")))
       .withColumn("kz", F.when(F.col("loo"), F.col("k_sz") - F.col("cz")).otherwise(F.col("k_sz")))
       .withColumn("kn", F.when(F.col("loo"), F.col("k_n_cp") - 1).otherwise(F.col("k_n_cp"))))
rn = F.sqrt(F.col("kx") ** 2 + F.col("ky") ** 2 + F.col("kz") ** 2)
P = (P.withColumn("_rn", rn)
      .withColumn("w", F.when((F.col("kn") >= MIN_CP_BW) & (F.col("_rn") > 0), width_km(F.col("_rn") / F.col("kn"))))
      .withColumn("ux", F.col("kx") / F.col("_rn")).withColumn("uy", F.col("ky") / F.col("_rn")).withColumn("uz", F.col("kz") / F.col("_rn"))
      .where((F.col("kn") >= MIN_CP_BW_TRUST) & F.col("w").isNotNull() & (F.col("w") >= COLOC_WIDTH_KM)))
r = (P.withColumn("_rk", F.row_number().over(Window.partitionBy("ent_id").orderBy("w", "j"))).where(F.col("_rk") <= K_MAX)
      .withColumn("iv", 1 / F.col("w") ** 2))
EST = (r.groupBy("ent_id").agg(F.sum(F.col("iv") * F.col("ux")).alias("ex"), F.sum(F.col("iv") * F.col("uy")).alias("ey"),
                               F.sum(F.col("iv") * F.col("uz")).alias("ez"), F.sum("iv").alias("siv"),
                               F.count("*").alias("k_used"), F.sum(F.col("loo").cast("int")).alias("neighbours_left_out"))
         .withColumn("_n", F.sqrt(F.col("ex") ** 2 + F.col("ey") ** 2 + F.col("ez") ** 2))
         .withColumn("pred_width_km", 1 / F.sqrt("siv")))
CAL = pd.read_csv(f"{GEO_DIR}/radius_calibration.csv").sort_values("wbin")
b = F.lit(int(CAL.wbin.iloc[-1]))
for _, row in CAL.iloc[::-1].iterrows():
    b = F.when(F.col("pred_width_km") <= float(row.width_to_km), F.lit(int(row.wbin))).otherwise(b)
CALDF = F.broadcast(spark.createDataFrame(CAL[["wbin", "r50_km", "r90_km", "p_within_50km", "evidence_class"]]))
EST = EST.withColumn("wbin", b).join(CALDF, "wbin", "left")
pub = (spark.table(T_CPTY_GEO).select(F.col("ent_id").cast("string").alias("ent_id"), F.col("est_lat").alias("p_lat"),
                                      F.col("est_lon").alias("p_lon"), F.col("evidence_class").alias("pub_class"),
                                      F.col("r50_km").alias("pub_r50"), F.col("r90_km").alias("pub_r90")))
px, py, pz = unit(F.col("p_lat"), F.col("p_lon"))
# footprint prior and core states, as in pkg_geo_metrics
gl = spark.table(T_CUST_GEO).where("geo_ok").select("lat", "lon", "state")
gx, gy, gz = unit(F.col("lat"), F.col("lon"))
fp = gl.agg(F.sum(gx).alias("x"), F.sum(gy).alias("y"), F.sum(gz).alias("z")).first()
_fn = math.sqrt(fp.x ** 2 + fp.y ** 2 + fp.z ** 2)
sc = gl.where(F.col("state").isNotNull()).groupBy("state").count().toPandas().sort_values("count", ascending=False)
sc["cum"] = sc["count"].cumsum() / sc["count"].sum()
CORE = sc.loc[sc["cum"].shift(fill_value=0) < CORE_SHARE, "state"].tolist()
V = (MATCH.join(EST, "ent_id", "left").join(pub, "ent_id", "left").join(DIRECT, "ent_id", "left")
       .withColumn("truth_tier", F.when(F.coalesce("direct_link", F.lit(False)), "a: customer traded with the entity")
                   .when(F.col("name_exact"), "b: identical full name").otherwise("c: cleaned name only"))
       .withColumn("err", F.when(F.col("_n") > 0, chord_km(F.col("ex") / F.col("_n"), F.col("ey") / F.col("_n"), F.col("ez") / F.col("_n"),
                                                          F.col("cx"), F.col("cy"), F.col("cz"))))
       .withColumn("err_pub", F.when(F.col("p_lat").isNotNull(), chord_km(px, py, pz, F.col("cx"), F.col("cy"), F.col("cz"))))
       .withColumn("err_prior", chord_km(F.lit(fp.x / _fn), F.lit(fp.y / _fn), F.lit(fp.z / _fn), F.col("cx"), F.col("cy"), F.col("cz")))
       .withColumn("evidence_class", F.coalesce("evidence_class", F.lit("NO_EVIDENCE")))
       .withColumn("pin", F.when(F.col("pin_representative"), "representative").when(F.col("pin_representative").isNotNull(), "not representative")
                   .otherwise("no flow centre"))
       .withColumn("footprint", F.when(F.col("state").isin(CORE), "in core states").otherwise("outside core states")))
V.write.mode("overwrite").parquet(f"{WORK}/v"); V = spark.read.parquet(f"{WORK}/v")
def score(df, by, err="err", r50="r50_km", r90="r90_km", p50="p_within_50km"):
    d = df.where(F.col(err).isNotNull())
    a = [F.count("*").alias("entities"), F.expr(f"percentile_approx({err}, 0.5)").alias("median_km"),
         F.expr(f"percentile_approx({err}, 0.9)").alias("p90_km"),
         F.avg((F.col(err) <= 25).cast("double")).alias("hit_25"), F.avg((F.col(err) <= 50).cast("double")).alias("hit_50"),
         F.avg((F.col(err) <= 250).cast("double")).alias("hit_250"),
         F.avg((F.col(err) <= F.col(r50)).cast("double")).alias("inside_r50"), F.avg((F.col(err) <= F.col(r90)).cast("double")).alias("inside_r90"),
         F.avg((F.col(err) > FAR_KM).cast("double")).alias(f"over_{FAR_KM}km"),
         F.expr("percentile_approx(err_prior, 0.5)").alias("prior_median_km")]
    if p50:
        a.append(F.avg(p50).alias("predicted_p50"))
    return d.groupBy(*by).agg(*a).toPandas()
ORD = {c: i for i, c in enumerate(CLS)}
S_CLS = score(V, ["evidence_class"]).assign(_o=lambda d: d.evidence_class.map(ORD)).sort_values("_o").drop(columns="_o")
S_PUB = score(V.withColumnRenamed("evidence_class", "_e").withColumnRenamed("pub_class", "evidence_class"),
              ["evidence_class"], err="err_pub", r50="pub_r50", r90="pub_r90", p50=None)
S_PUB = S_PUB.assign(_o=lambda d: d.evidence_class.map(ORD)).sort_values("_o").drop(columns="_o")
LOC = V.where(F.col("evidence_class").isin("PIN", "METRO"))
S_PIN = score(LOC, ["pin"])
S_FOOT = score(LOC, ["footprint"])
S_TIER = score(V.where(F.col("evidence_class").isin("PIN", "METRO")), ["truth_tier", "evidence_class"]).sort_values(["truth_tier", "evidence_class"], ascending=[True, False])
TIERS = V.groupBy("truth_tier").agg(F.count("*").alias("matched"), F.sum(F.col("err").isNotNull().cast("long")).alias("with_estimate")).toPandas().sort_values("truth_tier")
cov = V.agg(F.count("*").alias("n"), F.sum(F.col("err").isNotNull().cast("long")).alias("est"),
            F.sum(F.col("evidence_class").isin("PIN", "METRO").cast("long")).alias("loc"),
            F.sum((F.col("neighbours_left_out") > 0).cast("long")).alias("loo")).first()
for n_, d_ in [("outside_by_class_loo", S_CLS), ("outside_by_class_published", S_PUB), ("outside_pin_metro_by_truth", S_PIN),
               ("outside_pin_metro_by_footprint", S_FOOT), ("outside_pin_metro_by_truth_tier", S_TIER)]:
    d_.to_csv(f"{OUT_DIR}/{n_}.csv", index=False)
pct = ("hit_25", "hit_50", "hit_250", "inside_r50", "inside_r90", "predicted_p50", f"over_{FAR_KM}km")
out(sec(f"§4 Outside test · {cov['n']:,} matched entities · estimate for {cov['est']:,} · PIN or METRO {cov['loc']:,} · "
        f"matched customer removed from at least one neighbour's cloud for {cov['loo'] or 0:,} · {(time.time() - t0) / 60:.1f} min"),
    para("Read: inside_r50 near 50% and inside_r90 near 90% mean the radii hold for outside entities; hit_50 near "
         "predicted_p50 means the class probabilities hold. Compare with the customer self-scoring in pkg_geo_metrics §4."),
    sec("By evidence class, leave-one-out estimate"), tbl(S_CLS, pct=pct),
    sec("By evidence class, published estimate (pkg_cpty_geo)"), tbl(S_PUB, pct=pct),
    sec("PIN and METRO only, by ground-truth quality of the matched customer"), tbl(S_PIN, pct=pct),
    sec(f"PIN and METRO only, by footprint (core states: {', '.join(CORE)})"), tbl(S_FOOT, pct=pct),
    sec("Truth tiers (a: strongest evidence that the customer is the same business)"), tbl(TIERS),
    sec("PIN and METRO by truth tier: if tier a holds 50 / 90 coverage, the gap is wrong truth, not the radii"), tbl(S_TIER, pct=pct))

## §5 Strict leave-one-out on a sample of customers (v1.1)

`pkg_geo_metrics` scores customer t by removing only t from each neighbour j's cloud. Other records at the same spot
(affiliates of the same business, a household, the same building) stay in the cloud and give the answer away. Here a
`STRICT_PCT`% sample of located customers is scored twice with the same neighbours: the standard way, and with every
cloud point within `STRICT_KM` of t removed. Both go through the same calibration bins. If the strict version's
coverage of r50 / r90 falls well below 50 / 90%, the published radii are too tight and `pkg_geo_metrics` §4 should
calibrate with the strict leave-one-out (geo v1.2; only §4–§5 rerun).

In [ ]:
# §5  Strict leave-one-out
t0 = time.time()
G = (spark.table(T_CUST_GEO).where("geo_ok").select(F.trim(F.col("mdm_id")).alias("m"), "lat", "lon", "geo_status"))
gx, gy, gz = unit(F.col("lat"), F.col("lon"))
G = (G.withColumn("mx", gx).withColumn("my", gy).withColumn("mz", gz)
       .withColumn("gi", F.floor(F.col("lat") / STRICT_CELL_DEG).cast("long")).withColumn("gj", F.floor(F.col("lon") / STRICT_CELL_DEG).cast("long")))
G.write.mode("overwrite").parquet(f"{WORK}/g"); G = spark.read.parquet(f"{WORK}/g")
CC = (E.where((F.col("src_kind") == "CUST") & (F.col("dst_kind") == "CUST"))
        .select(F.col("src").alias("a"), F.col("dst").alias("b")))
CC = CC.unionByName(CC.select(F.col("b").alias("a"), F.col("a").alias("b"))).distinct()      # a's cloud holds b
CC.write.mode("overwrite").parquet(f"{WORK}/cc"); CC = spark.read.parquet(f"{WORK}/cc")
T = (G.where(F.col("geo_status").isin(TRUTH_STATUSES) & (F.abs(F.xxhash64(F.concat(F.col("m"), F.lit(str(SEED))))) % 100 < STRICT_PCT))
       .select(F.col("m").alias("t"), F.col("mx").alias("tx"), F.col("my").alias("ty"), F.col("mz").alias("tz"), F.col("gi").alias("ti"), F.col("gj").alias("tj")))
TJ = CC.join(T, CC.b == T.t).select("t", F.col("a").alias("j"), "tx", "ty", "tz", "ti", "tj")      # j has t in its cloud
TJ = TJ.join(K, "j")
TJ.write.mode("overwrite").parquet(f"{WORK}/tj"); TJ = spark.read.parquet(f"{WORK}/tj")
# cloud points of j within STRICT_KM of t: j's counterparties in the 3 x 3 grid cells around t
cells = TJ.select("t", "j", "tx", "ty", "tz", F.explode(F.array(*[F.struct((F.col("ti") + di).alias("gi"), (F.col("tj") + dj).alias("gj"))
                                                                  for di in (-1, 0, 1) for dj in (-1, 0, 1)])).alias("c"))
cells = cells.select("t", "j", "tx", "ty", "tz", F.col("c.gi").alias("gi"), F.col("c.gj").alias("gj"))
cloud = CC.select(F.col("a").alias("j"), F.col("b").alias("m")).join(G.select("m", "mx", "my", "mz", "gi", "gj"), "m")
near = (cells.join(cloud, ["j", "gi", "gj"])
          .where(chord_km(F.col("mx"), F.col("my"), F.col("mz"), F.col("tx"), F.col("ty"), F.col("tz")) <= STRICT_KM)
          .groupBy("t", "j").agg(F.sum("mx").alias("nx"), F.sum("my").alias("ny"), F.sum("mz").alias("nz"), F.count("*").alias("nn")))
SP = (TJ.join(near, ["t", "j"], "left")
        .withColumn("nn", F.coalesce("nn", F.lit(0))).withColumn("nx", F.coalesce("nx", F.lit(0.0)))
        .withColumn("ny", F.coalesce("ny", F.lit(0.0))).withColumn("nz", F.coalesce("nz", F.lit(0.0))))
SP.write.mode("overwrite").parquet(f"{WORK}/sp"); SP = spark.read.parquet(f"{WORK}/sp")
def scored(mode):
    if mode == "standard":
        kx, ky, kz, kn = (F.col("k_sx") - F.col("tx"), F.col("k_sy") - F.col("ty"), F.col("k_sz") - F.col("tz"), F.col("k_n_cp") - 1)
    else:   # strict: every cloud point within STRICT_KM of t (t included)
        kx, ky, kz, kn = (F.col("k_sx") - F.col("nx"), F.col("k_sy") - F.col("ny"), F.col("k_sz") - F.col("nz"),
                          F.col("k_n_cp") - F.greatest(F.col("nn"), F.lit(1)))
    d = SP.select("t", "j", "tx", "ty", "tz", kx.alias("kx"), ky.alias("ky"), kz.alias("kz"), kn.alias("kn"))
    rn_ = F.sqrt(F.col("kx") ** 2 + F.col("ky") ** 2 + F.col("kz") ** 2)
    d = (d.withColumn("_rn", rn_)
          .withColumn("w", F.when((F.col("kn") >= MIN_CP_BW) & (F.col("_rn") > 0), width_km(F.col("_rn") / F.col("kn"))))
          .withColumn("ux", F.col("kx") / F.col("_rn")).withColumn("uy", F.col("ky") / F.col("_rn")).withColumn("uz", F.col("kz") / F.col("_rn"))
          .where((F.col("kn") >= MIN_CP_BW_TRUST) & F.col("w").isNotNull() & (F.col("w") >= COLOC_WIDTH_KM)))
    d = (d.withColumn("_rk", F.row_number().over(Window.partitionBy("t").orderBy("w", "j"))).where(F.col("_rk") <= K_MAX)
          .withColumn("iv", 1 / F.col("w") ** 2))
    e = (d.groupBy("t").agg(F.sum(F.col("iv") * F.col("ux")).alias("ex"), F.sum(F.col("iv") * F.col("uy")).alias("ey"),
                            F.sum(F.col("iv") * F.col("uz")).alias("ez"), F.sum("iv").alias("siv"),
                            F.first("tx").alias("tx"), F.first("ty").alias("ty"), F.first("tz").alias("tz"))
           .withColumn("_n", F.sqrt(F.col("ex") ** 2 + F.col("ey") ** 2 + F.col("ez") ** 2))
           .withColumn("pred_width_km", 1 / F.sqrt("siv"))
           .withColumn("err", chord_km(F.col("ex") / F.col("_n"), F.col("ey") / F.col("_n"), F.col("ez") / F.col("_n"), F.col("tx"), F.col("ty"), F.col("tz")))
           .withColumn("wbin", b).join(CALDF, "wbin", "left").withColumn("err_prior", F.lit(0.0)))
    r_ = score(e, ["evidence_class"])
    r_.insert(0, "leave_one_out", mode)
    return r_
STRICT = pd.concat([scored("standard"), scored("strict")])
STRICT["_o"] = STRICT.evidence_class.map(ORD); STRICT = STRICT.sort_values(["_o", "leave_one_out"]).drop(columns=["_o", "prior_median_km"])
ns = SP.agg(F.countDistinct("t").alias("t"), F.count("*").alias("pairs"), F.sum((F.col("nn") > 1).cast("long")).alias("extra"),
            F.sum(F.greatest(F.col("nn") - 1, F.lit(0))).alias("extra_pts")).first()
STRICT.to_csv(f"{OUT_DIR}/strict_leave_one_out.csv", index=False)
out(sec(f"§5 Strict leave-one-out · {ns['t']:,} sampled customers · {ns['pairs']:,} neighbour pairs · "
        f"{ns['extra'] or 0:,} pairs ({(ns['extra'] or 0) / max(ns['pairs'], 1):.1%}) hold other points within {STRICT_KM} km of the target "
        f"({ns['extra_pts'] or 0:,} points) · {(time.time() - t0) / 60:.1f} min"),
    para("Read: if 'strict' covers r50 / r90 clearly less than 'standard', the published radii are optimistic by that much."),
    tbl(STRICT, pct=pct))

## §6 Clean-up

In [ ]:
# §6  Clean-up
delete(WORK)
out(sec(f"§6 Work files deleted · CSV files in {OUT_DIR} · run time {(time.time() - T_START) / 60:.1f} min"))